# Deep Agents - Human-in-the-Loop

## 📚 학습 목표

이 노트북에서는 Deep Agents의 Human-in-the-Loop (HITL) 기능을 학습합니다.

### 주요 내용
1. **Human-in-the-Loop 개념** - 민감한 작업에 대한 사람의 승인이 필요한 이유
2. **기본 설정** - `interrupt_on` 매개변수를 사용한 도구 승인 설정
3. **Decision Types** - approve, edit, reject 결정 유형
4. **중단 처리** - 중단된 실행을 처리하고 재개하는 방법
5. **실전 예제** - 파일 삭제, 이메일 발송 등 실제 시나리오

---

## 🎯 Human-in-the-Loop란?

일부 도구 작업은 민감하거나 위험할 수 있으며 실행 전에 사람의 승인이 필요합니다.

**예시:**
- 파일 삭제
- 데이터베이스 수정
- 이메일 발송
- 금융 거래 실행

Deep Agents는 LangGraph의 interrupt 기능을 통해 이러한 워크플로우를 지원합니다.

## 1. 환경 설정 및 필수 라이브러리 임포트

In [ ]:
from dotenv import load_dotenv

# 환경 변수 로드
load_dotenv()


## 2. 도구(Tools) 정의

실습에 사용할 세 가지 도구를 정의합니다:
- `delete_file`: 파일 삭제 (민감한 작업)
- `read_file`: 파일 읽기 (일반 작업)
- `send_email`: 이메일 발송 (민감한 작업)

In [ ]:
from langchain_core.tools import tool

@tool
def delete_file(path: str) -> str:
    """파일 시스템에서 파일을 삭제합니다."""
    return f"{path} 파일이 삭제되었습니다."

@tool
def read_file(path: str) -> str:
    """파일 시스템에서 파일을 읽습니다."""
    return f"{path} 파일의 내용입니다: [파일 내용]"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """이메일을 발송합니다."""
    return f"{to}에게 이메일이 발송되었습니다.\n제목: {subject}"


## 3. Decision Types 이해하기

Human-in-the-Loop에서 사용할 수 있는 세 가지 결정 유형:

| Decision Type | 설명 | 사용 시나리오 |
|--------------|------|--------------|
| `approve` | 원래 인자로 도구 실행 | 에이전트 제안을 승인 |
| `edit` | 인자를 수정한 후 실행 | 일부 파라미터를 변경하고 싶을 때 |
| `reject` | 도구 호출을 건너뜀 | 실행하지 않기로 결정 |

### `interrupt_on` 설정 방법

```python
interrupt_on = {
    "delete_file": True,  # 기본값: approve, edit, reject 모두 허용
    "read_file": False,   # 중단 없음
    "send_email": {"allowed_decisions": ["approve", "reject"]},  # 편집 불가
}
```

## 4. 기본 Human-in-the-Loop Agent 생성

⚠️ **중요:** Human-in-the-Loop는 **반드시 checkpointer가 필요**합니다!

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from deepagents import create_deep_agent

# Checkpointer 생성 (HITL에 필수!)
checkpointer = MemorySaver()

# Deep Agent 생성
agent = create_deep_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[delete_file, read_file, send_email],
    interrupt_on={
        "delete_file": True,  # 모든 결정 허용 (approve, edit, reject)
        "read_file": False,   # 승인 불필요
        "send_email": {"allowed_decisions": ["approve", "reject"]},  # 편집 불가
    },
    checkpointer=checkpointer  # 필수!
)

print("Human-in-the-Loop Agent 생성 완료")

## 5. 실습 1: 기본 Interrupt 처리 (Approve)

**시나리오:**
1. 🤖 Agent 실행 → 중단 발생
2. 👤 사용자가 대기 중인 작업 확인
3. 👤 사용자가 결정을 내리고 Agent 재개
4. ✅ 최종 결과 확인

### Step 1: Agent 실행 및 중단 확인

In [ ]:
import uuid

# 새로운 thread 생성 (상태 유지를 위해)
config_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result_1 = agent.invoke({
    "messages": [{"role": "user", "content": "temp.txt 파일을 삭제해주세요."}]
}, config=config_1)

# Interrupt 확인 및 정보 출력
if result_1.get("__interrupt__"):
    print("\n실행이 중단되었습니다! 사용자 승인이 필요합니다.\n")
    
    interrupts_1 = result_1["__interrupt__"][0].value
    action_requests_1 = interrupts_1["action_requests"]
    review_configs_1 = interrupts_1["review_configs"]
    
    config_map_1 = {cfg["action_name"]: cfg for cfg in review_configs_1}
    
    # 대기 중인 작업 표시
    for action in action_requests_1:
        review_config = config_map_1[action["name"]]
        print(f"   도구: {action['name']}")
        print(f"   인자: {action['args']}")
        print(f"   허용된 결정: {review_config['allowed_decisions']}")
        print()
    
    print("다음 셀에서 결정을 내려주세요!")
else:
    print("중단 없이 실행 완료")

### Step 2: 사용자 결정 및 Agent 재개

아래 코드에서 `decisions_1` 부분을 수정하여 다른 결정을 내릴 수 있습니다.

In [ ]:
from langgraph.types import Command

# 여기서 결정을 변경할 수 있습니다:
# - {"type": "approve"} : 승인
# - {"type": "reject"} : 거부
# - {"type": "edit", "edited_action": {...}} : 수정
decisions_1 = [
    {"type": "approve"}  # 파일 삭제 승인
]

# Agent 재개
result_1 = agent.invoke(
    Command(resume={"decisions": decisions_1}),
    config=config_1  # 동일한 config 사용 필수!
)

# 최종 결과
print("\n" + "=" * 60)
print("최종 결과:")
print("=" * 60)
print(result_1["messages"][-1].content)

---

## 6. 실습 2: 도구 실행 거부 (Reject)

이번에는 중요한 파일 삭제를 **거부**해보겠습니다.

### Step 1: Agent 실행 및 중단 확인

In [ ]:
config_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result_2 = agent.invoke({
    "messages": [{"role": "user", "content": "important_data.txt 파일을 삭제해주세요."}]
}, config=config_2)

if result_2.get("__interrupt__"):
    print("\n실행이 중단되었습니다!\n")
    
    interrupts_2 = result_2["__interrupt__"][0].value
    action_requests_2 = interrupts_2["action_requests"]
    
    for action in action_requests_2:
        print(f"   도구: {action['name']}")
        print(f"   인자: {action['args']}")
        print()
    
    print("중요한 파일입니다! 다음 셀에서 결정해주세요.")
else:
    print("중단 없이 실행 완료")

### Step 2: 사용자 결정 (거부)

In [ ]:
from langgraph.types import Command

decisions_2 = [
    {"type": "reject"}  # 파일 삭제 거부
]

result_2 = agent.invoke(
    Command(resume={"decisions": decisions_2}),
    config=config_2
)

print("=" * 60)
print("최종 결과:")
print("=" * 60)
print(result_2["messages"][-1].content)

---

## 7. 실습 3: 도구 인자 편집 (Edit)

파일 이름을 변경하여 실행해보겠습니다.

### Step 1: Agent 실행 및 중단 확인

In [ ]:
config_3 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result_3 = agent.invoke({
    "messages": [{"role": "user", "content": "old_file.txt 파일을 삭제해주세요."}]
}, config=config_3)

if result_3.get("__interrupt__"):
    print("\n실행이 중단되었습니다!\n")
    
    interrupts_3 = result_3["__interrupt__"][0].value
    action_request_3 = interrupts_3["action_requests"][0]
    review_configs_3 = interrupts_3["review_configs"]
    
    print(f"원래 도구 호출:")
    print(f"   도구: {action_request_3['name']}")
    print(f"   인자: {action_request_3['args']}")
    print(f"   허용된 결정: {review_configs_3[0]['allowed_decisions']}")
    print()
    print("다음 셀에서 파일 이름을 변경해보세요!")
else:
    print("중단 없이 실행 완료")

### Step 2: 사용자 결정 (인자 수정)

In [ ]:
from langgraph.types import Command

print("파일 이름을 'old_file.txt'에서 'really_old_file.txt'로 변경합니다.\n")

# 인자를 수정하여 다른 파일 삭제
decisions_3 = [{
    "type": "edit",
    "edited_action": {
        "name": action_request_3["name"],  # 도구 이름 포함 필수
        "args": {"path": "really_old_file.txt"}  # 수정된 인자
    }
}]

print(f"수정된 인자: {decisions_3[0]['edited_action']['args']}\n")

result_3 = agent.invoke(
    Command(resume={"decisions": decisions_3}),
    config=config_3
)

print("=" * 60)
print("최종 결과:")
print("=" * 60)
print(result_3["messages"][-1].content)

---

## 8. 실습 4: 여러 도구 호출 처리

에이전트가 여러 도구를 동시에 호출할 때 각각에 대한 결정을 내리는 방법

### Step 1: Agent 실행 및 중단 확인

In [ ]:
config_4 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result_4 = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "temp.txt 파일을 삭제하고 [email protected]에게 '작업 완료' 제목으로 이메일을 보내주세요."
    }]
}, config=config_4)

if result_4.get("__interrupt__"):
    print("\n실행이 중단되었습니다! 여러 도구가 승인 대기 중입니다.\n")
    
    interrupts_4 = result_4["__interrupt__"][0].value
    action_requests_4 = interrupts_4["action_requests"]
    
    print(f"대기 중인 도구 수: {len(action_requests_4)}\n")
    
    # 각 도구에 대한 정보 표시
    for idx, action in enumerate(action_requests_4, 1):
        print(f"[{idx}] 도구: {action['name']}")
        print(f"    인자: {action['args']}")
        print()
    
    print("다음 셀에서 각 도구에 대한 결정을 내려주세요!")
else:
    print("중단 없이 실행 완료")

### Step 2: 사용자 결정 (여러 도구에 대해)

**중요:** decisions 리스트의 순서는 action_requests 순서와 일치해야 합니다!

In [ ]:
from langgraph.types import Command

print("=" * 60)
print("사용자 결정:")
print("=" * 60)
print("   [1] delete_file → APPROVE (승인)")
print("   [2] send_email → REJECT (거부)")
print()

# 각 도구에 대한 결정 (순서대로!)
decisions_4 = [
    {"type": "approve"},  # 첫 번째 도구: delete_file
    {"type": "reject"}    # 두 번째 도구: send_email
]

result_4 = agent.invoke(
    Command(resume={"decisions": decisions_4}),
    config=config_4
)

print("=" * 60)
print("최종 결과:")
print("=" * 60)
print(result_4["messages"][-1].content)

---

## 9. 실습 5: 이메일 발송 (편집 불가 설정)

`send_email`은 `["approve", "reject"]`만 허용하도록 설정했습니다. (편집 불가!)

### Step 1: Agent 실행 및 중단 확인

In [ ]:
config_5 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result_5 = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "[email protected]에게 '회의 알림' 제목으로 내일 회의 일정을 알리는 이메일을 보내주세요."
    }]
}, config=config_5)

if result_5.get("__interrupt__"):
    print("\n실행이 중단되었습니다!\n")
    
    interrupts_5 = result_5["__interrupt__"][0].value
    action_requests_5 = interrupts_5["action_requests"]
    review_configs_5 = interrupts_5["review_configs"]
    
    config_map_5 = {cfg["action_name"]: cfg for cfg in review_configs_5}
    
    for action in action_requests_5:
        review_config = config_map_5[action["name"]]
        print(f"   도구: {action['name']}")
        print(f"   인자: {action['args']}")
        print(f"   허용된 결정: {review_config['allowed_decisions']}")
        print(f"   편집은 불가능합니다!")
        print()
    
    print("다음 셀에서 승인 또는 거부를 선택하세요!")
else:
    print("중단 없이 실행 완료")

### Step 2: 사용자 결정 (승인 또는 거부만 가능)

In [ ]:
from langgraph.types import Command

# send_email은 편집이 불가능하므로 approve 또는 reject만 가능
decisions_5 = [
    {"type": "approve"}  # 이메일 발송 승인
]

result_5 = agent.invoke(
    Command(resume={"decisions": decisions_5}),
    config=config_5
)

print("\n" + "=" * 60)
print("최종 결과:")
print("=" * 60)
print(result_5["messages"][-1].content)

---

## 11. Best Practices (모범 사례)

### ✅ 1. 항상 Checkpointer 사용

```python
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
agent = create_deep_agent(
    tools=[...],
    interrupt_on={...},
    checkpointer=checkpointer  # HITL에 필수!
)
```

### ✅ 2. 동일한 Thread ID 사용

```python
# 첫 번째 호출
config = {"configurable": {"thread_id": "my-thread"}}
result = agent.invoke(input, config=config)

# 재개 시 (동일한 config 사용!)
result = agent.invoke(Command(resume={...}), config=config)
```

### ✅ 3. 결정 순서를 작업 순서와 일치

```python
if result.get("__interrupt__"):
    interrupts = result["__interrupt__"][0].value
    action_requests = interrupts["action_requests"]
    
    # 각 작업에 대해 순서대로 결정 생성
    decisions = []
    for action in action_requests:
        decision = get_user_decision(action)  # 사용자 로직
        decisions.append(decision)
    
    result = agent.invoke(
        Command(resume={"decisions": decisions}),
        config=config
    )
```

### ✅ 4. 위험도에 따른 설정

```python
interrupt_on = {
    # 고위험: 모든 옵션 허용
    "delete_file": {"allowed_decisions": ["approve", "edit", "reject"]},
    "send_email": {"allowed_decisions": ["approve", "edit", "reject"]},
    
    # 중위험: 편집 불가
    "write_file": {"allowed_decisions": ["approve", "reject"]},
    
    # 저위험: 중단 없음
    "read_file": False,
    "list_files": False,
}
```

---

## 12. 헬퍼 함수: Interrupt 정보 출력

재사용 가능한 헬퍼 함수를 만들어 Interrupt 정보를 깔끔하게 출력할 수 있습니다.

In [ ]:
def display_interrupt_info(result):
    """
    Interrupt 정보를 깔끔하게 출력하는 헬퍼 함수
    """
    if not result.get("__interrupt__"):
        return None
    
    interrupts = result["__interrupt__"][0].value
    action_requests = interrupts["action_requests"]
    review_configs = interrupts["review_configs"]
    
    config_map = {cfg["action_name"]: cfg for cfg in review_configs}
    
    print("\n실행이 중단되었습니다! 다음 작업들이 승인 대기 중입니다:\n")
    
    for idx, action in enumerate(action_requests, 1):
        review_config = config_map[action["name"]]
        print(f"[{idx}] 도구: {action['name']}")
        print(f"    인자: {action['args']}")
        print(f"    허용된 결정: {review_config['allowed_decisions']}")
        print()
    
    return {
        "interrupts": interrupts,
        "action_requests": action_requests,
        "review_configs": review_configs,
        "config_map": config_map
    }

# 헬퍼 함수 테스트
config_helper = {"configurable": {"thread_id": str(uuid.uuid4())}}

result_helper = agent.invoke({
    "messages": [{"role": "user", "content": "test.txt 파일을 삭제해주세요."}]
}, config=config_helper)

# 헬퍼 함수로 정보 출력
interrupt_info = display_interrupt_info(result_helper)

if interrupt_info:
    print("이제 다음 셀에서 결정을 내리고 Agent를 재개하면 됩니다!")
    print("   예: decisions = [{'type': 'approve'}]")

---

## 13. 요약 및 핵심 포인트

### 🎯 Human-in-the-Loop의 핵심

1. **Checkpointer 필수**: HITL는 반드시 checkpointer가 필요합니다.
2. **Thread ID 일관성**: 중단과 재개 시 동일한 thread_id를 사용해야 합니다.
3. **결정 순서**: decisions 리스트는 action_requests 순서와 일치해야 합니다.

### 📊 Decision Types

| Type | 설명 | 사용 시나리오 |
|------|------|--------------|
| approve | 원래 인자로 실행 | 에이전트 제안 승인 |
| edit | 인자 수정 후 실행 | 파라미터 변경 필요 |
| reject | 실행 건너뛰기 | 작업 취소 |

### 🛡️ 보안 모범 사례

```python
interrupt_on = {
    # 고위험 작업: 모든 컨트롤
    "delete_file": {"allowed_decisions": ["approve", "edit", "reject"]},
    
    # 중위험 작업: 편집 불가
    "send_email": {"allowed_decisions": ["approve", "reject"]},
    
    # 저위험 작업: 승인 불필요
    "read_file": False,
}
```

### 🚀 실전 활용 시나리오

- **파일 시스템 작업**: 삭제, 수정 등 민감한 작업
- **외부 API 호출**: 이메일 발송, 결제 처리
- **데이터베이스 작업**: 삭제, 업데이트
- **시스템 명령어**: 서버 재시작, 설정 변경

---

## 🎓 학습 완료!

이제 Deep Agents에서 Human-in-the-Loop를 효과적으로 활용할 수 있습니다.

### 다음 단계
- 실제 프로젝트에 HITL 적용하기
- 커스텀 승인 로직 구현하기
- UI/UX와 통합하여 사용자 친화적인 승인 시스템 만들기